# 03. Trasformare un dataset QA in dati per decision model

Questo e il notebook centrale della prima fase. Prendiamo due dataset pubblici gratuiti e convertiamo ogni esempio in un insieme di domande tipizzate con supervisione.

PubMedQA contiene domande biomediche con etichetta yes/no/maybe e una risposta lunga. Banking77 contiene brevi messaggi di clienti con una tra 77 etichette di intento.

L'output e JSONL secondo lo schema in `data/SCHEMA.md`. Ogni cella di codice gestisce un passo e ogni passo termina con una validazione.


<div align='center'><img src='../images/qa_to_decision_dataset.png' alt='Da QA al decision dataset' style='width:80%; max-width:850px;'></div>

*Una riga QA diventa diverse domande tipizzate. Il file JSONL è il ponte tra il lavoro sui dati di questo notebook e training e valutazione.*


## Passo 1. Caricare una piccola fetta di PubMedQA

Il loader usa il dataset pubblico di Hugging Face quando la rete e disponibile. In caso contrario ripiega sui piccoli campioni inclusi in `data/samples`. In entrambi i casi l'esercizio e gratuito e riproducibile.


In [ ]:
from decision_qa import load_examples
pubmed = load_examples('pubmedqa', split='train', limit=8)
print(len(pubmed))


In [ ]:
first_pm = pubmed[0]
print(first_pm.id)
print(first_pm.label)


In [ ]:
print(first_pm.question)
print(first_pm.answer[:120])


## Passo 2. Caricare una piccola fetta di Banking77

Banking77 e un dataset di classificazione di intenti. Ogni riga ha un breve messaggio e un nome di etichetta come `card_arrival` o `exchange_rate`.


In [ ]:
banking = load_examples('banking77', split='train', limit=8)
print(len(banking))


In [ ]:
first_bk = banking[0]
print(first_bk.context)
print(first_bk.label)


## Passo 3. Impostare le opzioni di conversione

La configurazione controlla quanti distractor campionare, il seed casuale per la riproducibilita e la lunghezza massima del contesto. Un seed fisso e importante: rende il dataset generato identico a ogni esecuzione.


In [ ]:
from decision_qa import ConversionConfig
cfg = ConversionConfig(n_distractors=3, seed=42)
print(cfg.seed, cfg.n_distractors)


In [ ]:
assert cfg.n_distractors > 0
print('conversion config OK')


## Passo 4. Convertire PubMedQA

Per ogni riga di PubMedQA generiamo quattro domande: due affermazioni Noul (una supportata e una contraddetta), una Choice su yes/no/maybe e una Score sulla scala di supporto.


In [ ]:
from decision_qa import convert_pubmedqa
pm_records = convert_pubmedqa(pubmed, cfg, split='train')
print(len(pm_records))


In [ ]:
pm0 = pm_records[0]
print(pm0.id, pm0.split)
print(pm0.state['question'])


In [ ]:
print([q.type for q in pm0.questions])
assert len(pm0.questions) == 4


## Passo 5. Esaminare la coppia Noul

Ogni record deve contenere un'affermazione vera e una falsa. Questo insegna al modello entrambi i lati del giudizio, non solo il caso positivo.


In [ ]:
from decision_qa.questions import NoulQuestion
nouls = [q for q in pm0.questions if isinstance(q, NoulQuestion)]
print([q.gold for q in nouls])


In [ ]:
assert sorted(q.gold for q in nouls) == [False, True]
print('Noul pair has both polarities')


In [ ]:
for q in nouls:
    print(q.label, q.statement[:80])


## Passo 6. Esaminare la domanda Choice

Le tre decisioni vengono mescolate. Verifichiamo che l'indice gold punti ancora all'etichetta originale dopo il mescolamento.


In [ ]:
from decision_qa.questions import ChoiceQuestion
pm_choice = next(q for q in pm0.questions if isinstance(q, ChoiceQuestion))
print(pm_choice.options)


In [ ]:
print(pm_choice.options[pm_choice.gold_index])
assert 0 <= pm_choice.gold_index < len(pm_choice.options)


## Passo 7. Esaminare la domanda Score

La scala di supporto e ordinata da contradicted a supported. Ogni record mostra nella rubrica o la risposta gold o una contraddetta, quindi il livello gold alterna tra i due estremi. Questo da al modello un obiettivo reale invece di un'etichetta costante.


In [ ]:
from decision_qa.questions import ScoreQuestion
pm_score = next(q for q in pm0.questions if isinstance(q, ScoreQuestion))
print(pm_score.levels)


In [ ]:
print(pm_score.rubric[:120])
print(pm_score.levels[pm_score.gold_index])


In [ ]:
score_golds = [next(q for q in r.questions if isinstance(q, ScoreQuestion)).gold_index for r in pm_records]
print(sorted(set(score_golds)))
assert set(score_golds) == {0, 2}


## Passo 8. Convertire Banking77

Per ogni messaggio l'intento corretto viene mescolato con distractor campionati dall'intero insieme di 77 nomi di etichetta, non solo dalla piccola fetta caricata. Usiamo hard negative lessicali perche il task non sia banale.


In [ ]:
from decision_qa import convert_banking77
bk_records = convert_banking77(banking, cfg, split='train')
print(len(bk_records))


In [ ]:
bk0 = bk_records[0]
bk_choice = next(q for q in bk0.questions if isinstance(q, ChoiceQuestion))
print(bk_choice.options)


In [ ]:
print(bk_choice.options[bk_choice.gold_index])
assert bk0.state['context'] == first_bk.context


## Passo 9. Verificare i distractor

Un distractor non deve mai essere l'etichetta corretta. Altrimenti la domanda avrebbe due risposte corrette e la supervisione sarebbe sbagliata.


In [ ]:
gold_label = bk_choice.options[bk_choice.gold_index]
distractors = [o for i, o in enumerate(bk_choice.options) if i != bk_choice.gold_index]
print(distractors)


In [ ]:
assert gold_label not in distractors
print('no duplicate gold among distractors')


## Passo 10. Scrivere e leggere il JSONL

`build_dataset` carica, converte, divide e scrive il file. Poi lo rileggiamo e contiamo i record per split per confermare che tutti e tre siano presenti.


In [ ]:
from decision_qa import build_dataset, read_jsonl
path = build_dataset('pubmedqa', 'data/pubmedqa_decisions.jsonl', limits={'train': 40, 'val': 8, 'test': 8}, cfg=cfg)
print(path.name)


In [ ]:
records = read_jsonl(path)
print(len(records))


In [ ]:
from collections import Counter
counts = Counter(r.split for r in records)
print(dict(counts))
assert set(counts) == {'train', 'val', 'test'}


Un grafico rapido mostra l'equilibrio dei tipi di domanda. Un dataset con sole domande Choice addestrerebbe un solo paradigma, quindi la miscela conta.


In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
type_counts = Counter(q.type for record in records for q in record.questions)
labels = list(type_counts)
values = list(type_counts.values())
fig, ax = plt.subplots(figsize=(6.4, 3.4), constrained_layout=True)
ax.bar(labels, values, color=['#4C78A8', '#F58518', '#54A24B'])
ax.set_ylabel('Questions', labelpad=8)
ax.set_title('Question types in the converted dataset')
ax.grid(axis='y', alpha=0.25)
ax.set_ylim(0, max(values) * 1.2)
for index, value in enumerate(values):
    ax.text(index, value + max(values) * 0.02, str(value), ha='center', fontsize=9)
plt.show()


## Passo 11. Validare l'intero dataset

Verifichiamo due invarianti su ogni record: l'indice gold di ogni Choice e dentro la lista di opzioni, e ogni affermazione Noul non e vuota. Questi controlli intercettano la maggior parte dei bug di conversione.


In [ ]:
checked = 0
for record in records:
    for question in record.questions:
        if isinstance(question, ChoiceQuestion):
            assert 0 <= question.gold_index < len(question.options)
        if isinstance(question, NoulQuestion):
            assert question.statement.strip()
        checked += 1
print(checked, 'questions checked')


## Passo 12. Anteprima del payload di richiesta

Infine ritraduciamo un record convertito nel formato di rete. Si chiude cosi il cerchio: testo QA grezzo entra, domande tipizzate pronte per un decision model escono.


In [ ]:
from decision_qa import to_systemone_payload
import json
preview = to_systemone_payload(records[0].state, records[0].questions)
print(json.dumps(preview, indent=2)[:600])


In [ ]:
assert len(preview['questions']) == 4
print('payload ready for a decision model')


## Prossimi passi

Il notebook 04 fa il fine-tuning di un piccolo modello open su questo JSONL. Il notebook 05 valuta le probabilita. Lo schema e documentato in `data/SCHEMA.md`.
